# Vígil.ia — Melhoria do RT-DETR (v3): blur + balanceamento + multi-grão sintético

Notebook **autossuficiente** (não depende de células de outros notebooks).
Parte do RT-DETR **base** já treinado (12.5k) e refaz o fine-tune com dados melhores:

1. **Motion blur sintético** — cada foto de treino ganha uma cópia borrada → ensina a ler defeito em movimento (no vídeo, defeito borrado virava tudo "quebrado").
2. **Balanceamento de classes** — oversampling dos defeitos (o real tinha 122 intact vs ~21–32 por defeito).
3. **Compositor multi-grão** — cola grãos recortados (máscara por saturação) em fundos escuros/cinza → ~600 cenas de "esteira" perfeitamente rotuladas, de graça, com grãos encostando.

**Juiz do resultado:** re-rodar o MESMO `teste_soja.mp4` e comparar com o vídeo anterior
(mesmo vídeo = teste A/B limpo). A validação de 40 fotos quase não muda — o ganho é no vídeo.

Pré-requisitos no Drive: fotos reais (pastas por classe) e o peso base `soja_rtdetr_base.pt`.

## 1. Setup

In [ ]:
!pip -q install "ultralytics==8.4.80"

import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'Sem GPU!'
print('GPU:', torch.cuda.get_device_name(0),
      f'| livre: {torch.cuda.mem_get_info()[0]/1e9:.0f} GB')

## 2. Caminhos

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Fotos reais (pastas com nomes PT das classes; busca recursiva)
REAL_SRCS = [
    '/content/drive/MyDrive/Soja total/Soja total/Lotes',
    '/content/drive/MyDrive/Soja pra completar',
]

# Peso do RT-DETR base (treinado nas 12.5k). Usa o local se a sessão for a mesma;
# senão cai no backup do Drive.
BASE_PT = '/content/runs/detect/runs_rtdetr/base_12k/weights/best.pt'
if not os.path.exists(BASE_PT):
    BASE_PT = '/content/drive/MyDrive/soja_rtdetr_base.pt'
assert os.path.exists(BASE_PT), 'Peso base não encontrado! Confira o backup no Drive.'
print('base:', BASE_PT)

# Vídeo de teste (o MESMO do teste anterior, p/ comparação A/B)
VIDEO_TESTE = '/content/drive/MyDrive/teste_soja.mp4'

## 3. Funções (coleta, segmentação, blur, compositor, builder)

In [ ]:
import glob, hashlib, unicodedata, cv2, yaml
import numpy as np

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']
ALIASES = {0: ['broken', 'quebrad'], 1: ['immature', 'imatur', 'nao maduro'],
           2: ['intact'], 3: ['skin', 'casca', 'ardid', 'danific'], 4: ['spotted', 'manchad']}
IGNORE = ['part of the original']
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
RNG = np.random.default_rng(42)

def norm(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()

def class_of(folder):
    n = norm(folder)
    if any(norm(k) in n for k in IGNORE):
        return None
    for idx in range(5):
        if any(norm(k) in n for k in ALIASES[idx]):
            return idx
    return None

def collect_real(srcs, val_frac=0.15):
    items = []
    for src in srcs:
        for root, _, files in os.walk(src):
            cls = None
            for part in reversed(root.split(os.sep)):
                c = class_of(part)
                if c is not None:
                    cls = c; break
            if cls is None:
                continue
            for fn in files:
                if fn.lower().endswith(IMG_EXT):
                    p = os.path.join(root, fn)
                    h = int(hashlib.md5(p.encode()).hexdigest(), 16)
                    items.append((p, cls, 'val' if (h % 100) < val_frac * 100 else 'train'))
    from collections import Counter
    print('coletado:', dict(Counter(sp for _, _, sp in items)))
    return items

# ---------- segmentação (caixa do grão) ----------

def sat_box(img):
    """PRINCIPAL — Otsu no canal de SATURAÇÃO: grão amarelado vs fundo cinza.
    Imune a sombra/gradiente de brilho."""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def otsu_box(img):
    """RESERVA — Otsu no brilho (bom em fundo preto)."""
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    if area < 0.005 * h * w or area > 0.995 * h * w:
        return None
    x, y, bw, bh = cv2.boundingRect(c)
    pad = int(0.04 * min(bw, bh)) + 2
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + bw + pad), min(h, y + bh + pad)
    return (((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h, (x2 - x1) / w, (y2 - y1) / h)

def letterbox640(img, size=640):
    """Resize + borda preta centrada. Devolve escala e offsets p/ mapear a caixa.
    (A caixa é SEMPRE calculada na imagem original — com a borda antes, fundo
    cinza virava 'foreground' e a caixa saía do tamanho da foto inteira.)"""
    h, w = img.shape[:2]
    s = size / max(h, w)
    img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))))
    h, w = img.shape[:2]
    top, left = (size - h) // 2, (size - w) // 2
    img = cv2.copyMakeBorder(img, top, size - h - top, left, size - w - left,
                             cv2.BORDER_CONSTANT, value=(0, 0, 0))
    return img, s, left, top

# ---------- melhorias v3 ----------

def motion_blur(img, rng=RNG):
    """Borrão de movimento sintético: kernel linear em ângulo aleatório."""
    k = int(rng.choice([7, 9, 11, 13, 15]))
    kernel = np.zeros((k, k), np.float32)
    kernel[k // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), float(rng.uniform(0, 180)), 1)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    kernel /= max(kernel.sum(), 1e-6)
    return cv2.filter2D(img, -1, kernel)

def extract_cutout(img):
    """Recorta o grão COM máscara (p/ o compositor)."""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    s = cv2.GaussianBlur(hsv[:, :, 1], (5, 5), 0)
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    area = cv2.contourArea(c)
    h, w = img.shape[:2]
    if area < 0.01 * h * w or area > 0.90 * h * w:
        return None
    mask = np.zeros((h, w), np.uint8)
    cv2.drawContours(mask, [c], -1, 255, -1)
    x, y, bw, bh = cv2.boundingRect(c)
    return img[y:y + bh, x:x + bw], mask[y:y + bh, x:x + bw]

def make_scene(cutouts, rng=RNG, size=640):
    """Cola 6–25 grãos recortados num fundo escuro/cinza — cena de esteira
    sintética perfeitamente rotulada (alguns grãos podem encostar)."""
    bg = int(rng.integers(20, 130))
    canvas = np.clip(np.full((size, size, 3), bg, np.int16)
                     + rng.normal(0, 6, (size, size, 3)), 0, 255).astype(np.uint8)
    occ = np.zeros((size, size), np.uint8)
    boxes = []
    for _ in range(int(rng.integers(6, 26))):
        cls, crop, mask = cutouts[int(rng.integers(len(cutouts)))]
        s = int(rng.integers(60, 150)) / max(crop.shape[:2])
        crop2 = cv2.resize(crop, None, fx=s, fy=s)
        mask2 = cv2.resize(mask, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
        h2, w2 = crop2.shape[:2]
        diag = int(np.ceil(np.hypot(h2, w2))) + 2
        M = cv2.getRotationMatrix2D((w2 / 2, h2 / 2), float(rng.uniform(0, 360)), 1)
        M[0, 2] += (diag - w2) / 2
        M[1, 2] += (diag - h2) / 2
        crop3 = cv2.warpAffine(crop2, M, (diag, diag))
        mask3 = cv2.warpAffine(mask2, M, (diag, diag), flags=cv2.INTER_NEAREST)
        ys, xs = np.where(mask3 > 0)
        if not len(xs):
            continue
        crop3 = crop3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        mask3 = mask3[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        gh, gw = mask3.shape
        if gh >= size - 2 or gw >= size - 2:
            continue
        placed = False
        for _try in range(20):  # acha vaga (leve sobreposição = grãos encostando)
            px = int(rng.integers(0, size - gw))
            py = int(rng.integers(0, size - gh))
            inter = (occ[py:py + gh, px:px + gw] > 0) & (mask3 > 0)
            if inter.sum() <= 0.15 * (mask3 > 0).sum():
                placed = True
                break
        if not placed:
            continue
        alpha = (cv2.GaussianBlur(mask3, (5, 5), 0).astype(np.float32) / 255)[..., None]
        reg = canvas[py:py + gh, px:px + gw]
        canvas[py:py + gh, px:px + gw] = (alpha * crop3 + (1 - alpha) * reg).astype(np.uint8)
        occ[py:py + gh, px:px + gw][mask3 > 0] = 255
        boxes.append((cls, (px + gw / 2) / size, (py + gh / 2) / size, gw / size, gh / size))
    return canvas, boxes

def balance_train(items):
    """Oversampling: duplica itens das classes minoritárias até igualar a maior."""
    from collections import defaultdict
    train = [it for it in items if it[2] == 'train']
    rest = [it for it in items if it[2] != 'train']
    by = defaultdict(list)
    for it in train:
        by[it[1]].append(it)
    mx = max(len(v) for v in by.values())
    out = []
    for c, v in by.items():
        out += v + [v[int(i)] for i in RNG.integers(0, len(v), mx - len(v))]
    print('balanceado (train):', {NAMES[c]: sum(1 for it in out if it[1] == c) for c in sorted(by)})
    return out + rest

def build_v3(items, out_dir, n_synth=600, blur_frac=0.4):
    assert items, 'Nenhuma imagem coletada! Confira REAL_SRCS.'
    for sp in ('train', 'val', 'test'):
        os.makedirs(f'{out_dir}/images/{sp}', exist_ok=True)
        os.makedirs(f'{out_dir}/labels/{sp}', exist_ok=True)
    items = balance_train(items)
    cutouts = []
    kept = skipped = 0
    for i, (path, cls, sp) in enumerate(items):
        if i % 200 == 0:
            print(f'  fotos {i}/{len(items)}…', flush=True)
        img = cv2.imread(path)
        if img is None:
            skipped += 1; continue
        h0, w0 = img.shape[:2]
        box = sat_box(img) or otsu_box(img)
        if box is None:
            skipped += 1; continue
        if sp == 'train':
            cut = extract_cutout(img)
            if cut is not None:
                cutouts.append((cls, cut[0], cut[1]))
        lb, s, left, top = letterbox640(img)
        cx, cy, ww, hh = box
        cx = (cx * w0 * s + left) / 640.0
        cy = (cy * h0 * s + top) / 640.0
        ww = (ww * w0 * s) / 640.0
        hh = (hh * h0 * s) / 640.0
        line = f'{cls} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}'
        stem = f'{sp}_{i:06d}'
        cv2.imwrite(f'{out_dir}/images/{sp}/{stem}.jpg', lb, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/{sp}/{stem}.txt', 'w').write(line)
        kept += 1
        if sp == 'train':  # cópia com borrão de movimento (mesma caixa)
            cv2.imwrite(f'{out_dir}/images/train/{stem}b.jpg', motion_blur(lb),
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            open(f'{out_dir}/labels/train/{stem}b.txt', 'w').write(line)
            kept += 1
    print(f'fotos reais: kept={kept} skipped={skipped} | banco de recortes: {len(cutouts)} grãos')
    assert cutouts, 'Nenhum recorte extraído!'
    synth = 0
    for j in range(n_synth):
        if j % 100 == 0:
            print(f'  cenas {j}/{n_synth}…', flush=True)
        canvas, boxes = make_scene(cutouts)
        if not boxes:
            continue
        if RNG.random() < blur_frac:
            canvas = motion_blur(canvas)
        stem = f'synth_{j:05d}'
        cv2.imwrite(f'{out_dir}/images/train/{stem}.jpg', canvas, [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{out_dir}/labels/train/{stem}.txt', 'w').write(
            '\n'.join(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}' for c, cx, cy, w, h in boxes))
        synth += 1
    print(f'cenas sintéticas: {synth}')
    yaml.safe_dump({'path': out_dir, 'train': 'images/train', 'val': 'images/val',
                    'test': 'images/test', 'names': {i: n for i, n in enumerate(NAMES)}},
                   open(f'{out_dir}/data.yaml', 'w'), sort_keys=False, allow_unicode=True)
    return f'{out_dir}/data.yaml'

## 4. Construir o dataset v3

In [ ]:
V3_YAML = build_v3(collect_real(REAL_SRCS), '/content/soja_det_v3')
print('data.yaml ->', V3_YAML)

## 5. Sanidade visual — OLHAR ANTES DE TREINAR
Cenas sintéticas: grãos bem colados, caixas justas, fundo convincente.

In [ ]:
import matplotlib.pyplot as plt

def show_labeled(paths, title):
    plt.figure(figsize=(13, 8))
    for i, p in enumerate(paths[:6]):
        img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        for ln in open(p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'):
            c, cx, cy, ww, hh = ln.split()
            cx, cy, ww, hh = float(cx), float(cy), float(ww), float(hh)
            cv2.rectangle(img, (int((cx - ww / 2) * w), int((cy - hh / 2) * h)),
                          (int((cx + ww / 2) * w), int((cy + hh / 2) * h)), (0, 255, 0), 2)
        ax = plt.subplot(2, 3, i + 1); ax.imshow(img); ax.axis('off')
    plt.suptitle(title); plt.tight_layout(); plt.show()

show_labeled(sorted(glob.glob('/content/soja_det_v3/images/train/synth_*.jpg')), 'Cenas sintéticas')
show_labeled(sorted(glob.glob('/content/soja_det_v3/images/train/train_*b.jpg')), 'Fotos reais com blur')

## 6. Retreino (fine-tune v3, a partir do base)
`amp=False` + `warmup_epochs=0` são OBRIGATÓRIOS: o warmup padrão (bias-lr 0.1)
colapsa um modelo já convergido em dataset pequeno, e RT-DETR+AMP pode dar NaN.

In [ ]:
from ultralytics import RTDETR

ft3 = RTDETR(BASE_PT)
ft3.train(
    data=V3_YAML, epochs=40, imgsz=640, batch=12, device=0, seed=42,
    optimizer='AdamW', lr0=5e-5, patience=15, close_mosaic=8,
    amp=False, warmup_epochs=0.0,
    cache=False, workers=8,
    mosaic=1.0, hsv_v=0.5, degrees=15, translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.5,
    project='runs_rtdetr', name='ft_v3', exist_ok=True,
)

## 7. Backup do peso no Drive (SEMPRE, antes de qualquer teste)

In [ ]:
!cp /content/runs/detect/runs_rtdetr/ft_v3/weights/best.pt /content/drive/MyDrive/soja_rtdetr_ft_v3.pt
print('backup ok: soja_rtdetr_ft_v3.pt')

## 8. Teste A/B no MESMO vídeo
Compara com o vídeo anotado da rodada anterior, principalmente nos trechos de
movimento médio (onde defeito virava tudo "quebrado").

In [ ]:
best = RTDETR('/content/runs/detect/runs_rtdetr/ft_v3/weights/best.pt')
best.predict(source=VIDEO_TESTE, conf=0.25, save=True,
             project='runs_rtdetr', name='pred_video_v3', exist_ok=True)

!cp /content/runs/detect/runs_rtdetr/pred_video_v3/* /content/drive/MyDrive/
print('vídeo anotado copiado pro Drive ✅')

## 9. Bônus: rastreamento + voto temporal (sem treino)
Cada grão ganha um ID persistente (ByteTrack) e a classe final é o voto
majoritário ao longo dos frames — corrige a classificação nos frames borrados.
É assim que sistemas industriais reais estabilizam a leitura.

In [ ]:
from collections import Counter, defaultdict

votes = defaultdict(Counter)  # id do grão -> contagem de classes ao longo do vídeo
for r in best.track(source=VIDEO_TESTE, conf=0.25, tracker='bytetrack.yaml',
                    stream=True, persist=True, verbose=False):
    if r.boxes.id is None:
        continue
    for tid, c in zip(r.boxes.id.int().tolist(), r.boxes.cls.int().tolist()):
        votes[tid][NAMES[c]] += 1

print(f'grãos rastreados: {len(votes)}')
intact = 0
for tid, cnt in sorted(votes.items()):
    cls, n = cnt.most_common(1)[0]
    total = sum(cnt.values())
    if cls == 'intact':
        intact += 1
    print(f'  grão #{tid:3d}: {cls:13s} ({n}/{total} frames)')
pct = intact / len(votes) if votes else 0
print(f'\nQC premium (voto temporal): {intact}/{len(votes)} intactos ({pct*100:.0f}%)'
      f' -> {"APROVADO" if pct >= 0.9 else "REPROVADO"}')